In [1]:
from dotenv import load_dotenv
import os
import shutil
import pandas as pd
import sys
import openpyxl as opxl
import logging

import dictionary as dic
from models import Database
load_dotenv()

db = Database(
    host=os.environ["HOST"],
    port=os.environ["PORT"],
    database=os.environ["DATABASE"],
    user=os.environ["USER"],
    password=os.environ["PASSWORD"]
    )

In [1]:
input_path = r'C:\Users\kongh\Downloads\19DECBARCODE RELABELED.xlsx'
output_path = r'C:\Users\kongh\Downloads\19DECBARCODE RELABELED_NoReturned.xlsx'

In [2]:
pur = db.select(table='purchase')[['pur_id','pur_gold_cost','pur_code','pur_date']]
# stk = pd.read_excel(r'C:\Users\keong\Downloads\NoReturnStock.xlsx')
stk = pd.read_excel(input_path,sheet_name='Sheet1')
stk = stk.loc[stk['stk_returned']==0]
stk["pur_code"] = stk["pur_code"].str.replace(" ", "", regex=False).str.upper()
stk_to_insert = pd.merge(stk,pur,how='left',on='pur_code').drop(['pur_code'],axis=1)
stk_to_insert[['stk_status']] = 'IN STOCK'
stk_to_insert.rename({'pur_id':'stk_pur_id','pur_gold_cost':'stk_gold_cost','pur_date':'stk_pur_date'},axis=1,inplace=True)
# stk_to_insert.drop(['Old_tag'],axis=1,inplace=True)
# stk_to_insert.to_excel(r'C:\Users\keong\Downloads\12July2025_Bracelet_completed.xlsx',index=False)
stk_to_insert.to_excel(output_path,index=False)

NameError: name 'db' is not defined

## Check Distinct Pur Code

In [16]:
import pandas as pd
pur_code = pd.read_excel(r'C:\Users\keong\Downloads\distinctPur_code.xlsx')
pur_code["pur_code"] = pur_code["pur_code"].str.replace(r"[ _]", "", regex=True).str.upper()
pur = db.select(table='purchase')[['pur_id','pur_gold_cost','pur_code','pur_date']]
pur_code["exists_in_pur"] = pur_code["pur_code"].isin(pur["pur_code"])

In [17]:
pur_code.to_excel(r'C:\Users\keong\Downloads\distinctPur_code.xlsx',index=False)

## Barcode template Generator

In [4]:
df = pd.read_excel(r'C:\Users\kongh\Downloads\STOCK BARCODE 22SEP2025.xlsx')
df.drop('stk_id',axis=1,inplace=True)
df['stk_barcode_text'] = df['stk_barcode']
df['stk_weight'] = df['stk_weight'].apply(lambda x: f"{x:.2f}G")
df['stk_pur_monthyear'] = df['stk_pur_monthyear'].astype(str).apply(lambda x: x.zfill(4) if len(x) == 3 else x).str.lstrip("'")

def format_bracketed(x):
    return f"({int(x)})" if x == int(x) else f"({x})"

df['stk_length_size'] = df['stk_length_size'].apply(format_bracketed)
# Create new DataFrame where even-indexed rows are merged with previous ones
new_rows = []
for i in range(0, len(df), 2):
    row = df.iloc[i].copy()
    if i + 1 < len(df):
        row2 = df.iloc[i + 1].copy()
        row2.index = [col + '_2' for col in row2.index]
        combined = pd.concat([row, row2])
        new_rows.append(combined)
    else:
        new_rows.append(row)  # if odd number of rows, just append last row

new_df = pd.DataFrame(new_rows).reset_index(drop=True)
new_df.to_excel(r'C:\Users\kongh\Downloads\STOCK BARCODE 22SEP2025_relabeled.xlsx',index=False)